# Fase 3.3 a 3.5 — Testes, associação e regressão

**Pergunta central:** o concurso público estoura mais o custo por causa do procedimento,
ou porque as obras de concurso são maiores?

O notebook segue em três passos:

1. **Testes de comparação (P3, P4):** as diferenças entre procedimentos e faixas de porte são reais ou acaso? E são grandes?
2. **Associação custo × prazo (P5):** correlação de Spearman.
3. **Regressão logística:** a probabilidade de estouro em função de porte, procedimento, tipo de obra, região, ano e concorrência, todos ao mesmo tempo.

Janela: empreitadas celebradas de 2019 a 2022 com CPIc válido (D6).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

RAIZ = Path("..")
FIG = RAIZ / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200)

e = pd.read_parquet(RAIZ / "data" / "processed" / "empreitadas.parquet")
e["estouro"] = (e["CPIc"] < 1).astype(int)
e["desvio_custo_pct"] = (1 / e["CPIc"] - 1) * 100

PROCEDIMENTOS = {"Ajuste Direto Regime Geral": "Ajuste direto",
                 "Consulta Prévia": "Consulta prévia",
                 "Concurso público": "Concurso público"}
e["procedimento"] = e["tipoprocedimento"].map(PROCEDIMENTOS).fillna("Outros")

FAIXAS = [0, 150_000, 1_000_000, 5_000_000, np.inf]
e["faixa_porte"] = pd.cut(e["precoContratual"], FAIXAS,
                          labels=["até 150 mil", "150 mil a 1 M", "1 a 5 M", "acima de 5 M"])

d = e[e["ano_celebracao"].between(2019, 2022) & e["CPIc"].notna()].copy()
print(f"Contratos na análise: {len(d):,}")

## 1. Testes de comparação entre grupos

**Qui-quadrado:** testa se a percentagem de estouro é igual em todos os grupos.
Um p-valor pequeno (abaixo de 0,05) indica que a diferença dificilmente é acaso.

**V de Cramér:** mede o *tamanho* da diferença, de 0 (nenhuma) a 1 (total).
Com mais de 20 mil contratos, quase qualquer diferença dá um p-valor minúsculo, então
o tamanho do efeito é o que diz se a diferença importa. Referência usual para tabelas
com 2 categorias de resposta: cerca de 0,1 é pequeno, 0,3 é médio e 0,5 é grande.

In [ ]:
def qui_quadrado(df, grupo):
    tabela = pd.crosstab(df[grupo], df["estouro"])
    qui2, p, gl, _ = stats.chi2_contingency(tabela)
    v = np.sqrt(qui2 / (tabela.values.sum() * (min(tabela.shape) - 1)))
    pct = (100 * df.groupby(grupo, observed=True)["estouro"].mean()).round(1)
    print(f"--- Estouro de custo por {grupo}")
    print(pct.to_string())
    print(f"Qui-quadrado = {qui2:,.1f} | graus de liberdade = {gl} | p-valor = {p:.2g} | V de Cramér = {v:.3f}\n")

principais = d[d["procedimento"] != "Outros"]
qui_quadrado(principais, "procedimento")
qui_quadrado(d, "faixa_porte")

**Mann-Whitney:** entre os contratos que *estouraram*, o tamanho do estouro é diferente
no concurso público e no ajuste direto? É um teste para dados assimétricos, que compara
as distribuições sem supor normalidade. O tamanho de efeito é a probabilidade de
superioridade: a chance de um estouro de concurso, sorteado ao acaso, ser maior que um
de ajuste direto (0,5 = nenhuma diferença).

In [ ]:
estouros = d[d["estouro"] == 1]
concurso = estouros.loc[estouros["procedimento"] == "Concurso público", "desvio_custo_pct"]
ajuste = estouros.loc[estouros["procedimento"] == "Ajuste direto", "desvio_custo_pct"]
u, p = stats.mannwhitneyu(concurso, ajuste, alternative="two-sided")
print(f"Estouro mediano: concurso {concurso.median():.1f}% | ajuste direto {ajuste.median():.1f}%")
print(f"Mann-Whitney U = {u:,.0f} | p-valor = {p:.2g} | probabilidade de superioridade = {u / (len(concurso) * len(ajuste)):.3f}")

## 2. Custo e prazo estouram juntos? (P5)

**Correlação de Spearman** entre o desvio de custo e o tempo até o fecho em relação ao prazo
(1 ÷ SPIc_fecho). Spearman usa a ordem dos valores, e não os valores em si, por isso é
robusta a assimetrias e a extremos. Vai de −1 a +1; perto de 0 = sem associação.
Por usar razões, e não dias, compara obras curtas e longas na mesma escala.

In [ ]:
cp = d[d["SPIc_fecho"].notna()]
rho, p = stats.spearmanr(cp["desvio_custo_pct"], 1 / cp["SPIc_fecho"])
print(f"Desvio de custo × (tempo até o fecho ÷ prazo): rho de Spearman = {rho:.3f} | p-valor = {p:.2g} | n = {len(cp):,}")

prorrogado = cp["prorrogacao_dias"].fillna(0) > 0
rho2, p2 = stats.spearmanr(cp["desvio_custo_pct"], prorrogado)
print(f"Desvio de custo × teve prorrogação formal:       rho de Spearman = {rho2:.3f} | p-valor = {p2:.2g}")

## 3. Regressão logística da probabilidade de estouro

A regressão estima o efeito de cada variável **mantendo as outras constantes**. É isso que
as tabelas por faixa só conseguiam fazer de forma grosseira.

**Variáveis explicativas**

| Variável | Como entra | Referência |
| --- | --- | --- |
| Porte | log10 do preço contratual a preços de 2021 (D10) | — |
| Prazo contratual | log10 dos dias | — |
| Procedimento | ajuste direto, consulta prévia, concurso público, outros | ajuste direto |
| Tipo de obra | divisão CPV (3 dígitos); as raras juntas em "outros" | a mais frequente |
| Região | distrito do local de execução; os raros juntos em "outros" | Lisboa |
| Ano de celebração | 2019 a 2022 | 2019 |
| Concorrência | nº de concorrentes em faixas, com "não informado" | 1 concorrente |

**Erros-padrão agrupados por entidade adjudicante:** contratos do mesmo dono de obra
não são independentes (mesma equipa, mesmas práticas). Agrupar evita intervalos de
confiança estreitos demais.

**Três modelos, em camadas:** (1) só o procedimento; (2) + porte e prazo; (3) + todos os controles.
Se o efeito do concurso público encolher do modelo 1 para o 2, parte dele era porte.
Se ainda sobrar no modelo 3, existe um efeito próprio do procedimento.

In [ ]:
def agrupar_raros(serie, minimo=300, rotulo="outros"):
    contagem = serie.value_counts()
    return serie.where(serie.isin(contagem[contagem >= minimo].index), rotulo).fillna(rotulo)

m = d[d["preco_contratual_2021"].gt(0) & d["prazoExecucao"].gt(0)].copy()
m["log_preco"] = np.log10(m["preco_contratual_2021"].astype(float))
m["log_prazo"] = np.log10(m["prazoExecucao"].astype(float))
m["cpv"] = agrupar_raros(m["cpv_divisao"])
m["distrito"] = agrupar_raros(m["local_distrito"])
m["ano"] = m["ano_celebracao"].astype(str)
m["concorrencia"] = (pd.cut(m["concorrentes"], [0, 1, 3, 5, np.inf], labels=["1", "2 a 3", "4 a 5", "6 ou mais"])
                      .cat.add_categories("não informado").fillna("não informado").astype(str))
m = m.dropna(subset=["estouro", "procedimento", "log_preco", "log_prazo", "cpv", "distrito", "ano", "concorrencia"])
m["entidade"] = m["adjudicante"].astype("string").str.extract(r"^(\d{9})", expand=False).fillna(m["adjudicante"])
grupos = pd.factorize(m["entidade"])[0]

cpv_ref = m["cpv"].value_counts().index[0]
dist_ref = "Lisboa" if "Lisboa" in set(m["distrito"]) else m["distrito"].value_counts().index[0]
proc = "C(procedimento, Treatment('Ajuste direto'))"
formulas = {
    "1. só procedimento": f"estouro ~ {proc}",
    "2. + porte e prazo": f"estouro ~ {proc} + log_preco + log_prazo",
    "3. + todos os controles": (f"estouro ~ {proc} + log_preco + log_prazo"
                                f" + C(cpv, Treatment('{cpv_ref}')) + C(distrito, Treatment('{dist_ref}'))"
                                f" + C(ano, Treatment('2019')) + C(concorrencia, Treatment('1'))"),
}
modelos = {nome: smf.logit(f, data=m).fit(disp=0, cov_type="cluster", cov_kwds={"groups": grupos})
           for nome, f in formulas.items()}
print(f"Contratos no modelo: {len(m):,} | entidades: {len(set(grupos)):,}")
for nome, mod in modelos.items():
    print(f"{nome:<26} pseudo-R² = {mod.prsquared:.3f} | AIC = {mod.aic:,.0f}")

### Razões de chances (odds ratios)

Uma razão de chances de 1,8 para o concurso público quer dizer: **as chances** de estouro no
concurso são 1,8 vez as do ajuste direto, mantendo as demais variáveis constantes.
Acima de 1 = mais estouro; abaixo de 1 = menos. Se o intervalo de 95% não incluir o 1,
o efeito é estatisticamente significativo.

In [ ]:
def razoes_chances(mod):
    ic = mod.conf_int()
    t = pd.DataFrame({"OR": np.exp(mod.params), "IC 2,5%": np.exp(ic[0]),
                      "IC 97,5%": np.exp(ic[1]), "p-valor": mod.pvalues})
    t.index = (t.index.str.replace(r"C\((\w+), Treatment\('[^']*'\)\)\[T\.(.+)\]", r"\1: \2", regex=True))
    return t

completo = razoes_chances(modelos["3. + todos os controles"])
completo.round(3)

### Como o efeito do concurso público muda entre os modelos

O gráfico mostra a razão de chances do concurso público (contra o ajuste direto) nos três modelos.

In [ ]:
COR = {"azul": "#2a78d6", "laranja": "#eb6834", "aqua": "#1baf7a",
       "texto": "#0b0b0b", "texto2": "#52514e", "grade": "#e4e3df", "fundo": "#fcfcfb"}
plt.rcParams.update({
    "figure.facecolor": COR["fundo"], "axes.facecolor": COR["fundo"], "axes.edgecolor": COR["grade"],
    "axes.labelcolor": COR["texto2"], "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlecolor": COR["texto"], "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "x", "grid.color": COR["grade"],
    "xtick.color": COR["texto2"], "ytick.color": COR["texto2"], "font.size": 10.5,
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 160, "savefig.bbox": "tight",
})

termos = {"Concurso público": COR["azul"], "Consulta prévia": COR["laranja"]}
linhas = []
for nome, mod in modelos.items():
    t = razoes_chances(mod)
    for termo in termos:
        linha = t.loc[f"procedimento: {termo}"]
        linhas.append({"modelo": nome, "termo": termo, **linha.to_dict()})
efeito = pd.DataFrame(linhas)

fig, ax = plt.subplots(figsize=(9, 4.4))
nomes = list(modelos)
for i, termo in enumerate(termos):
    g = efeito[efeito["termo"] == termo]
    y = np.arange(len(nomes))[::-1] + (0.15 if i == 0 else -0.15)
    ax.errorbar(g["OR"], y, xerr=[g["OR"] - g["IC 2,5%"], g["IC 97,5%"] - g["OR"]], fmt="o",
                color=termos[termo], ecolor=termos[termo], markersize=8, markeredgecolor=COR["fundo"],
                markeredgewidth=2, capsize=3, label=termo)
    for xi, yi in zip(g["OR"], y):
        ax.annotate(f"{xi:.2f}".replace(".", ","), (xi, yi), textcoords="offset points", xytext=(0, 9),
                    ha="center", color=COR["texto"], fontsize=9.5, zorder=4,
                    bbox=dict(boxstyle="round,pad=0.15", facecolor=COR["fundo"], edgecolor="none"))
ax.axvline(1, color=COR["texto2"], linewidth=1, linestyle="--")
ax.set_yticks(np.arange(len(nomes))[::-1], nomes)
ax.set_ylim(-0.55, len(nomes) - 0.25)
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.1f}".replace(".", ","))
ax.set_xlabel("Razão de chances de estouro (referência: ajuste direto = 1)")
ax.legend(loc="lower right")
oc1 = efeito.query("termo == 'Concurso público'")["OR"].iloc[0]
oc3 = efeito.query("termo == 'Concurso público'")["OR"].iloc[-1]
ax.set_title(f"Controlando o porte, o efeito do concurso público cai de {oc1:.1f} para {oc3:.1f}".replace(".", ","), pad=26)
ax.text(0, 1.02, "Razões de chances de estouro de custo, empreitadas 2019-2022, IC de 95% "
        "com erros agrupados por entidade", transform=ax.transAxes, color=COR["texto2"], fontsize=10)
fig.text(0, -0.03, "Fonte: Portal Base (IMPIC), dados abertos. Regressão logística.",
         color=COR["texto2"], fontsize=8.5)
fig.savefig(FIG / "05_efeito_concurso_modelos.png")
plt.show()
efeito.round(3)

### Efeitos em pontos percentuais

As razões de chances são difíceis de explicar a um público leigo. Os **efeitos marginais médios**
traduzem o modelo completo em pontos percentuais: quanto muda a probabilidade de estouro
com cada variável, em média, mantendo as demais constantes. É a forma de apresentar no post.

In [ ]:
mfx = modelos["3. + todos os controles"].get_margeff(at="overall").summary_frame()
mfx.index = mfx.index.str.replace(r"C\((\w+), Treatment\('[^']*'\)\)\[T\.(.+)\]", r"\1: \2", regex=True)
mfx = (mfx[["dy/dx", "Conf. Int. Low", "Cont. Int. Hi."]] * 100).round(1)
mfx.columns = ["efeito (p.p.)", "IC 2,5%", "IC 97,5%"]
mfx

### Diagnóstico do modelo

- **VIF (fator de inflação da variância):** mede se as variáveis contínuas se sobrepõem tanto
  que o modelo não consegue separar os efeitos. Abaixo de 5 é tranquilo; acima de 10 é problema.
- **AUC:** a capacidade do modelo de ordenar os contratos, pondo os que estouraram acima dos que não.
  0,5 = sorteio; 0,7 a 0,8 = aceitável; acima de 0,8 = bom. Aqui o objetivo é explicar, não prever,
  então uma AUC modesta não invalida os efeitos estimados.

In [ ]:
continuas = m[["log_preco", "log_prazo"]].assign(constante=1)
vif = pd.Series([variance_inflation_factor(continuas.values, i) for i in range(2)],
                index=["log_preco", "log_prazo"]).round(2)
print("VIF:"); print(vif.to_string())

prob = modelos["3. + todos os controles"].predict(m)
u, _ = stats.mannwhitneyu(prob[m["estouro"] == 1], prob[m["estouro"] == 0])
print(f"\nAUC do modelo completo: {u / ((m['estouro'] == 1).sum() * (m['estouro'] == 0).sum()):.3f}")

completo.round(3).to_csv(RAIZ / "reports" / "regressao_razoes_chances.csv", sep=";", decimal=",", encoding="utf-8-sig")
mfx.to_csv(RAIZ / "reports" / "regressao_efeitos_marginais.csv", sep=";", decimal=",", encoding="utf-8-sig")
print("Tabelas gravadas em reports/")